In [ ]:
#install repo library
!pip install kagglehub

## Notebook Setup

In [ ]:
# Import initial 3rd party libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import kagglehub
import json

# Configure Notebook
%matplotlib inline
plt.style.use('fivethirtyeight')
sns.set_context("notebook")
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Download latest version
path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")
print("Path to dataset files:", path)

In [ ]:
import os

for root, dirs, files in os.walk(path):
    print(f"Directory: {root}")
    for file in files:
        file_path = os.path.join(root, file)
        print(f"  File: {file_path}")
        print(file)
        

In [ ]:
import os
import warnings
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split

In [ ]:
# Load the metadata
df_meta = pd.read_csv(path+"\\movies_metadata.csv")
df_meta = df_meta[["id", "title"]]
df_meta["id"] = pd.to_numeric(df_meta["id"], errors="coerce") 
df_meta = df_meta.dropna(subset=["id"]).astype({"id": int}) 
df_meta = df_meta.dropna(subset=["title"]) 

# Load the metadata
df_ratings = pd.read_csv(path+"\\ratings.csv")
df_ratings = df_ratings[["userId", "movieId", "rating"]]
df_ratings = df_ratings.dropna(subset=["userId", "movieId", "rating"])
df_ratings["userId"] = df_ratings["userId"].astype(int) 
df_ratings["movieId"] = df_ratings["movieId"].astype(int) 
df_ratings["rating"] = df_ratings["rating"].astype(float) 
# If there are duplicate <userId, movieId> pairs, average them. Unlikely tho 
df_ratings = df_ratings.groupby(["userId", "movieId"], as_index=False)["rating"].mean()


df_links = pd.read_csv(path+"\\links.csv") 
df_links = df_links[["movieId", "tmdbId"]]
df_links["tmdbId"] = pd.to_numeric(df_links["tmdbId"], errors="coerce") 
df_links = df_links.dropna(subset=["tmdbId"]).astype({"tmdbId": int}) 

In [ ]:
df_ratings.head()

In [ ]:
# check user rating count
user_count = df_ratings.groupby('userId',as_index=False).count().drop(columns=['rating'])
# user_count = user_count['userId'].count()
user_count = user_count.rename(columns={"movieId":"num_user_ratings"})
user_count.head()


In [ ]:
print("Number of users", len(user_count['userId']))

In [ ]:
sns.scatterplot(user_count,x='userId',y='num_user_ratings')

In [ ]:
user_count = df_ratings['userId'].value_counts()
user_keep = user_count[user_count>=25]
user_keep = user_keep[user_count<300].index

In [ ]:
# do the same for the number of movie ratings

In [ ]:
# check movie rating count
rating_count = df_ratings.groupby('movieId',as_index=False).count().drop(columns=['rating'])
# user_count = user_count['userId'].count()
rating_count = rating_count.rename(columns={"userId":"num_ratings"})
rating_count.head()


In [ ]:
print("Number of movies", len(rating_count['movieId']))

In [ ]:
sns.scatterplot(rating_count,x='movieId',y='num_ratings')

In [ ]:
movie_count = df_ratings['movieId'].value_counts()
movie_keep = movie_count[movie_count>=50].index

In [ ]:
# Apply both filters
df_filtered = df_ratings[
    df_ratings['userId'].isin(user_keep) &
    df_ratings['movieId'].isin(movie_keep)
].copy()

print(df_filtered.keys())
print(f"Rows before : {len(df_ratings):,}")
print(f"After filter  : {len(df_filtered):,}")
print(f"Remaining users : {df_filtered['userId'].nunique():,}")
print(f"Remaining movies: {df_filtered['movieId'].nunique():,}")

In [ ]:
# Create dense integer codes for the filtered ids
# Need to for sparse matrix
df_filtered['user_idx']  = pd.Categorical(df_filtered['userId']).codes
df_filtered['movie_idx'] = pd.Categorical(df_filtered['movieId']).codes

n_users  = df_filtered['user_idx'].max() + 1
n_movies = df_filtered['movie_idx'].max() + 1

print(f"Sparse matrix shape: ({n_users}, {n_movies})")

In [ ]:
from scipy.sparse import coo_matrix

# Data is large and will overload pivot. Use sparse matrix. Coordinate format

user_item_coo = coo_matrix(
    (df_filtered['rating'], (df_filtered['user_idx'], df_filtered['movie_idx'])), shape=(n_users, n_movies)
)

# Compressed sparse row conversion to make reading faster for knn
user_item_sparse = user_item_coo.tocsr()      
print(f"Non-zero items: {user_item_sparse.nnz:,}")
print(f"Memory: {user_item_sparse.nnz * 12 / 1e6:.1f} MB")

In [ ]:
user_id_to_idx  = dict(zip(df_filtered['userId'], df_filtered['user_idx']))
idx_to_user_id  = {v: k for k, v in user_id_to_idx.items()}

movie_id_to_idx = dict(zip(df_filtered['movieId'], df_filtered['movie_idx']))
idx_to_movie_id = {v: k for k, v in movie_id_to_idx.items()}

In [ ]:
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=15)
model.fit(user_item_sparse)
print("KNN")

In [ ]:
def recommend(user_id, n_recs=5):
    if user_id not in user_id_to_idx:
        return f"User {user_id} not in filtered set"

    u_idx = user_id_to_idx[user_id] # user id

    # Find similar users
    distances, indices = model.kneighbors(
        user_item_sparse[u_idx].reshape(1, -1),
        n_neighbors=16               # +1 to drop self
    )
    sim_idx = indices[0][1:]          # drop self
    sim_cos = 1 - distances[0][1:]    # cosine similarity

    # Predict for unrated movies
    rated = set(user_item_sparse[u_idx].indices)
    pred = {}

    for m_idx in range(n_movies):
        if m_idx in rated:
            continue
        # ratings from similar users
        ratings = []
        weights = []
        for s_idx, w in zip(sim_idx, sim_cos):
            r = user_item_sparse[s_idx, m_idx]
            if r > 0:
                ratings.append(r)
                weights.append(w)
        if ratings:
            pred[m_idx] = np.average(ratings, weights=weights)

    # Get the top movies
    top = sorted(pred.items(), key=lambda x: x[1], reverse=True)[:n_recs]
    recs = []
    for m_idx, score in top:
        mid = idx_to_movie_id[m_idx]
        linkid = df_links[df_links['movieId']==mid]['tmdbId'].iloc[0]
        title = df_meta[df_meta['id'] == linkid]['title']
        recs.append({'id': mid, 'title': title, 'predicted_rating': round(score, 2)})

    return pd.DataFrame(recs)


In [ ]:
# Test
print(recommend(user_id=1, n_recs=20))